In [0]:
# Databricks notebook source
# ============================================================
# MASTER PIPELINE ORCHESTRATOR
# E-Commerce Customer Intelligence Platform
# Triggers: Workflow 1 (Bronze) → Workflow 2 (Gold) → Workflow 3 (ML)
# Silver jobs are triggered internally by the ETL raw_to_bronze notebook
# via trigger_jobs in each bronze job config
# ============================================================


In [0]:
# MAGIC %md
# MAGIC ## 🚀 E-Commerce Intelligence — Master Pipeline
# MAGIC 
# MAGIC | Workflow | Scope | Notes |
# MAGIC |---|---|---|
# MAGIC | Workflow 1 | Bronze | Silver triggered internally per bronze job config |
# MAGIC | Workflow 2 | Gold | Waves derived from `dependent_jobs` in each gold JSON |
# MAGIC | Workflow 3 | ML | Phase 1 training → Phase 2 inference |


In [0]:
# COMMAND ----------
import json
from pyspark.sql.functions import date_format, current_timestamp

BASE_PATH = "/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform"

# ── Shared partition for the entire pipeline run ──────────────────────
partition = str(spark.range(1).select(
    date_format(current_timestamp(), "yyyyMMddHHmmssSSS")
    .cast("bigint").alias("p")
).collect()[0]["p"])

print(f"{'='*60}")
print(f"  🚀 E-COMMERCE INTELLIGENCE PIPELINE")
print(f"  Partition  : {partition}")
print(f"  Base path  : {BASE_PATH}")
print(f"{'='*60}")


In [0]:
# COMMAND ----------
def run_workflow(name, notebook_path, timeout=7200):
    print(f"\n{'─'*60}")
    print(f"  Starting {name}...")
    print(f"{'─'*60}")
    try:
        result      = dbutils.notebook.run(notebook_path, timeout, {"partition": partition})
        result_dict = json.loads(result)
        if result_dict.get("status") == "SUCCESS":
            print(f"  ✅ {name} completed successfully")
            return True
        raise Exception(f"{name} returned non-success: {result}")
    except Exception as e:
        print(f"  ❌ {name} FAILED: {str(e)}")
        raise


In [0]:
# COMMAND ----------
# MAGIC %md ## Workflow 1 — Bronze (Silver triggered internally)


In [0]:
# COMMAND ----------
run_workflow(
    name          = "Workflow 1 — Bronze",
    notebook_path = f"{BASE_PATH}/orchestration/workflow_bronze",
    timeout       = 10800
)


In [0]:
# COMMAND ----------
# MAGIC %md ## Workflow 2 — Gold (waves from dependent_jobs in JSON)


In [0]:
# COMMAND ----------
run_workflow(
    name          = "Workflow 2 — Gold",
    notebook_path = f"{BASE_PATH}/orchestration/workflow_gold",
    timeout       = 7200
)


In [0]:
# COMMAND ----------
# MAGIC %md ## Workflow 3 — ML


In [0]:
# COMMAND ----------
run_workflow(
    name          = "Workflow 3 — ML",
    notebook_path = f"{BASE_PATH}/orchestration/workflow_ml",
    timeout       = 14400
)


In [0]:
# COMMAND ----------
print(f"\n{'='*60}")
print(f"  🎉 FULL PIPELINE COMPLETE")
print(f"  Partition : {partition}")
print(f"{'='*60}")
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "partition": partition}))
